# 14. RAG Fundamentals

**Tier:** Building with LLMs
**Estimated time:** 45 minutes
**Prerequisites:** 03, 12, 13
**Priority:** 🔴 Crucial — one of the most common production LLM patterns. *If skipped, revisit when:* n/a.
**Source material:** @sairahul1 — "20 AI Concepts You Must Understand in 2026" (https://x.com/sairahul1/status/2057740928908161461)

## What You'll Learn
- Why a model confidently invents facts about documents it never saw — and how retrieval fixes that
- The four-step pipeline: chunk → embed → retrieve → generate, built from scratch
- Where this notebook ends and the full `07_rag_learning/` suite begins

## Why This Matters
This is the smallest version of the single most common production LLM pattern. Almost every "AI over my company's data" product is some variant of this loop — get it solid here in ~40 lines, then see the advanced versions (reranking, hybrid search, multimodal, graph, agentic routing) already built out in `07_rag_learning/`.


## A model that's never read your handbook will still answer confidently

A model is a compression of the text it trained on (notebook 06) — it has no access to *your* company's internal handbook, your product's latest spec sheet, or last week's support tickets. Ask it something specific about those documents and it doesn't say "I don't have that document" by default; it pattern-matches to the closest thing it *does* know and answers fluently anyway. That's a hallucination: confident, well-formed, and wrong.

**Retrieval-Augmented Generation (RAG)** fixes this not by retraining the model, but by handing it the relevant text *at request time*, inside the context window from notebook 13. The model still doesn't "know" your handbook — but for this one request, it can read the right page before answering. The pipeline has four steps:

1. **Chunk** — split documents into pieces small enough to retrieve and fit in context.
2. **Embed** — turn each chunk into a vector (notebook 03) capturing its meaning.
3. **Retrieve** — embed the query the same way, find the chunks whose vectors are closest.
4. **Generate** — hand the retrieved chunks to the model as context (notebook 12's delimiter pattern) and ask it to answer only from them.

We'll use the **Helios Robotics** corpus already in this repo (`data/corpus/`) — real spec sheets and incident reports the base model has never seen.


In [ ]:
import sys, os
sys.path.insert(0, "..")   # so `import ragkit` works from 03_building/

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"

def ask(prompt, system="You are a helpful assistant.", max_tokens=200, model=TEACH_MODEL):
    if not HAS_ANTHROPIC:
        print("  [skipped: no ANTHROPIC_API_KEY]")
        return None
    try:
        import anthropic
        client = anthropic.Anthropic()
        msg = client.messages.create(model=model, max_tokens=max_tokens, system=system,
                                      messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    except Exception as e:
        print(f"  [skipped: API call failed — {type(e).__name__}: {str(e)[:150]}]")
        return None


## Step 1: Watch the hallucination happen

We ask about a real Helios product spec the base model has no way of knowing precisely.


In [ ]:
QUESTION = "What is the maximum continuous payload, in kg, of the HeliosArm V2?"

print("--- Answer WITHOUT any context (the model is guessing) ---")
print(ask(QUESTION))


## Step 2: Build the pipeline from scratch — chunk, embed, retrieve

We reuse `ragkit`'s chunking and embedding helpers (the same ones `07_rag_learning/01_naive_rag.ipynb` builds on) rather than reimplementing tokenization or `all-MiniLM-L6-v2` loading from zero.


In [ ]:
from ragkit.data import load_corpus, chunk_text
from ragkit.embeddings import embed, cosine_similarity

docs = load_corpus()   # every .txt file in data/corpus/
print(f"Loaded {len(docs)} source documents")

chunks = []
for doc in docs:
    for piece in chunk_text(doc["text"], chunk_size=200, overlap=40):
        chunks.append(piece)
print(f"Split into {len(chunks)} chunks of ~200 chars each")

chunk_vectors = embed(chunks)          # (n_chunks, 384)
print("Chunk embedding matrix shape:", chunk_vectors.shape)


In [ ]:
query_vector = embed([QUESTION])             # (1, 384)
similarities = cosine_similarity(query_vector, chunk_vectors)   # (n_chunks,)

top_k = 3
top_indices = similarities.argsort()[::-1][:top_k]

for rank, idx in enumerate(top_indices, 1):
    print(f"#{rank}  score={similarities[idx]:.3f}")
    print("   ", chunks[idx][:160].replace(chr(10), ' '))


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

order = np.argsort(similarities)[::-1][:15]
plt.figure(figsize=(8, 4))
plt.bar(range(len(order)), similarities[order], color=["#C44E52" if i in top_indices else "#4C72B0" for i in order])
plt.title(f'Similarity of top 15 chunks to: "{QUESTION[:40]}..."')
plt.xlabel("Chunk rank"); plt.ylabel("Cosine similarity")
plt.tight_layout(); plt.show()


*The red bars are the top-3 chunks we'll hand to the model; similarity drops off sharply after the most relevant chunks, which is exactly the signal retrieval relies on.*


## Why this is a toy embedder — and what production RAG uses

`all-MiniLM-L6-v2` (384 dimensions, from notebook 03) is great for *learning* RAG: it's small, runs offline, and needs no API key. But it's also a 2021-era, general-purpose model — its retrieval quality is noticeably weaker than what production systems actually use. If you'd seen the wrong chunks come back above, the lesson "RAG doesn't work" would have been the wrong takeaway — the real culprit would have been the embedder, not the RAG pattern itself.

As of mid-2026, the embedding landscape looks like this:

| Model | Type | Notes |
|---|---|---|
| **Voyage 4** (`voyage-3-large` stable) | API | Anthropic's recommended embedding partner — Anthropic ships no embedding model of its own |
| **Gemini Embedding** | API | Natively multimodal — text, image, video, audio in one shared space |
| **Cohere embed-v4** | API | Multimodal, variable output dimensions |
| **OpenAI text-embedding-3-large** | API | Solid, no longer top-tier on MTEB |

(MTEB v2 scores aren't comparable to the older v1 scores — and any leaderboard is a prior, not a guarantee, for your specific data.)

Because every notebook in this repo calls a single `ragkit.embeddings.embed()` function, switching the **entire pipeline above** to Voyage needs no code change here — just set `EMBED_BACKEND=voyage` and `VOYAGE_API_KEY` in `.env`. The cell below demonstrates this if a key is present, and skips cleanly otherwise.


In [ ]:
HAS_VOYAGE = bool(os.environ.get("VOYAGE_API_KEY"))

if not HAS_VOYAGE:
    print("  [skipped: no VOYAGE_API_KEY — set EMBED_BACKEND=voyage and VOYAGE_API_KEY in .env to run this cell]")
else:
    os.environ["EMBED_BACKEND"] = "voyage"
    # Re-import so ragkit.embeddings picks up the new backend in this process.
    import importlib
    import ragkit.config, ragkit.embeddings
    importlib.reload(ragkit.config)
    importlib.reload(ragkit.embeddings)
    from ragkit.embeddings import embed as voyage_embed, cosine_similarity as voyage_cosine_similarity

    voyage_chunk_vectors = voyage_embed(chunks, input_type="document")
    voyage_query_vector = voyage_embed([QUESTION], input_type="query")
    voyage_similarities = voyage_cosine_similarity(voyage_query_vector, voyage_chunk_vectors)

    voyage_top_indices = voyage_similarities.argsort()[::-1][:top_k]
    print(f"Voyage embedding matrix shape: {voyage_chunk_vectors.shape}  (vs. MiniLM's {chunk_vectors.shape})")
    for rank, idx in enumerate(voyage_top_indices, 1):
        print(f"#{rank}  score={voyage_similarities[idx]:.3f}")
        print("   ", chunks[idx][:160].replace(chr(10), ' '))


## Step 3: Generate — grounded, this time

We pass the retrieved chunks as context, delimited the way notebook 12 recommended, with an explicit instruction not to answer outside them.


In [ ]:
RAG_SYSTEM_PROMPT = (
    "Answer ONLY using the information inside <context> tags. "
    "If the answer isn't in the context, say so explicitly. Cite which chunk you used."
)

context_block = "\n\n".join(f"<context chunk={i}>\n{chunks[idx]}\n</context>" for i, idx in enumerate(top_indices))
grounded_prompt = f"{context_block}\n\nQuestion: {QUESTION}"

print("--- Answer WITH retrieved context (grounded) ---")
print(ask(grounded_prompt, system=RAG_SYSTEM_PROMPT))


## Where this notebook ends

This ~40-line pipeline is the mechanism. For production-grade variants — cross-encoder reranking, hybrid BM25+vector search, multimodal (image) retrieval, knowledge-graph traversal, and an agent that *routes* between retrieval strategies — see the existing **`07_rag_learning/`** suite (notebooks 00-07), which already builds all of these on this same Helios corpus. Notebook 15 next covers what changes when your chunk count goes from hundreds to millions.


## Exercises


In [ ]:
# Exercise 1 (Warm-up): Change top_k
# Task: Re-run retrieval with top_k = 1 and top_k = 5. Does the grounded answer's confidence
#       or correctness change? Why might MORE context sometimes hurt?
# Hint: More chunks means more chance of an irrelevant chunk diluting the model's attention —
#       this is the recall-vs-precision tradeoff notebook 15 names explicitly.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): Ask an unanswerable question
# Task: Ask a question the corpus genuinely doesn't cover (e.g. "what color options does the
#       HeliosArm V2 come in?"). Run it both without context and with the RAG pipeline. Does
#       the grounded version correctly say "not in the context" instead of guessing a color?
# Hint: This is the entire point of the RAG_SYSTEM_PROMPT instruction — without it, the model
#       may still guess even with irrelevant context in front of it.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): Multi-document grounding
# Task: Pick a question whose answer spans TWO different source documents (e.g. one about a
#       spec AND a related incident report). Check whether top_k=3 retrieval pulls chunks from
#       both documents, and whether the generated answer correctly combines them.
# Hint: Print each retrieved chunk's source filename (chunk-to-doc mapping isn't tracked above —
#       you'll need to track it when building `chunks`, similar to how ragkit's
#       build_chunked_corpus returns parallel metadata).

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
for k in [1, 5]:
    idxs = similarities.argsort()[::-1][:k]
    ctx = "\n\n".join(f"<context>{chunks[i]}</context>" for i in idxs)
    print(f"--- top_k={k} ---")
    print(ask(f"{ctx}\n\nQuestion: {QUESTION}", system=RAG_SYSTEM_PROMPT))
# With k=1 you may miss a needed detail; with k=5 you risk diluting attention with
# off-topic chunks — there's a sweet spot, not a "more is always better" rule.

# Exercise 2
unanswerable_q = "What color options does the HeliosArm V2 come in?"
print("--- no context ---")
print(ask(unanswerable_q))
qv = embed([unanswerable_q])
sims = cosine_similarity(qv, chunk_vectors)
idxs = sims.argsort()[::-1][:3]
ctx = "\n\n".join(f"<context>{chunks[i]}</context>" for i in idxs)
print("--- grounded ---")
print(ask(f"{ctx}\n\nQuestion: {unanswerable_q}", system=RAG_SYSTEM_PROMPT))
# Without context the model likely invents a color; grounded + the explicit "say so" rule
# should make it admit the context doesn't cover color options.

# Exercise 3
chunks_with_source = []
for doc in docs:
    for piece in chunk_text(doc["text"], chunk_size=200, overlap=40):
        chunks_with_source.append((piece, doc.get("metadata", {}).get("source", doc.get("title", "unknown"))))
multi_doc_q = "What payload does the HeliosArm V2 support, and was that ever an incident cause?"
qv2 = embed([multi_doc_q])
texts_only = [c for c, _ in chunks_with_source]
vecs2 = embed(texts_only)
sims2 = cosine_similarity(qv2, vecs2)
top = sims2.argsort()[::-1][:3]
for i in top:
    print(chunks_with_source[i][1], "->", chunks_with_source[i][0][:100])
```
</details>


## Key Takeaways
- Models hallucinate confidently about documents they never trained on — RAG fixes this at request time, not by retraining.
- The pipeline is four steps: chunk, embed, retrieve, generate — each one reuses a tool from an earlier notebook (notebook 02/03 embeddings, notebook 12 prompting, notebook 13's context budget).
- An explicit "answer only from context, otherwise say so" instruction is what separates a grounded answer from a context-flavored hallucination.
- `top_k` trades recall (find everything relevant) against precision (don't dilute the model's attention) — see notebook 15.
- This 40-line version is the mechanism; `07_rag_learning/` has the production variants already built.

## What's Next
Notebook **15 — Vector Databases** asks what changes once your chunk count goes from a few hundred to millions: brute-force cosine similarity stops scaling, and that's exactly the problem a vector database is built to solve.
